In [ ]:
# General
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import math
import geopandas as gpd

# Plotting
import matplotlib.pyplot as plt
import plotly
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio

In [ ]:
# function to get unique values
def unique(list1):
 
    # initialize a null list
    unique_list = []
 
    # traverse for all elements
    for x in list1:
        # check if exists in unique_list or not
        if x not in unique_list:
            unique_list.append(x)
    return unique_list

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths

# SharePoint
path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'Census Data')
path_main = os.path.join(path_sp, 'Data')

# Git
path_git  = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
path_config = os.path.join(path_git, 'Python Code', 'Census', 'aa_config')

In [ ]:
# Import objects
df_params = pd.read_excel(os.path.join(path_config, 'ACS Configuration File.xlsx'), sheet_name = 'Inputs')

# Set parameters for querying ACS data
indicator_name     = df_params[df_params['Type'] == 'indicator_name'    ]['Input'].values[0]
estimate           = df_params[df_params['Type'] == 'estimate'          ]['Input'].values[0]
sample_type        = df_params[df_params['Type'] == 'sample'            ]['Input'].values[0]
workbook           = df_params[df_params['Type'] == 'geography'         ]['Input'].values[0]
import_tab         = df_params[df_params['Type'] == 'import_tab'        ]['Input'].values[0]
percentages        = df_params[df_params['Type'] == 'percentages'       ]['Input'].values[0]
population_weights = df_params[df_params['Type'] == 'population_weights']['Input'].values[0]
num_vars           = df_params[df_params['Type'] == 'num_vars'          ]['Input'].values[0]
year_start         = df_params[df_params['Type'] == 'year_start'        ]['Input'].values[0]
year_end           = df_params[df_params['Type'] == 'year_end'          ]['Input'].values[0]
report_theme       = df_params[df_params['Type'] == 'report_theme'      ]['Input'].values[0]
sp_folder_out      = df_params[df_params['Type'] == 'sp_folder'         ]['Input'].values[0]
import_tab = 'MPO'


# view
print(indicator_name)
print(estimate)
print(sample_type)
print(workbook)
print(import_tab)
print("Percentages: " + percentages)
print("Population weights: " + population_weights)
print("Number of variables: " + str(num_vars))
print(year_start)
print(year_end)

## Import Data

In [ ]:
if sample_type == 'ACS':
    df_acs = pd.read_excel(os.path.join(path_main, report_theme, sp_folder_out, indicator_name,
                                      indicator_name + ' ' + workbook + ' ' + estimate + ' Long.xlsx')
                          , sheet_name = import_tab
                          )
if sample_type == 'PUMS':
    df_acs = pd.read_excel(os.path.join(path_main, report_theme, sp_folder_out, indicator_name,
                                      indicator_name + ' ' + workbook + ' ' + estimate + ' Householders_SACOG vs DRCOG.xlsx')
                          )

df_acs.columns = [x.lower() for x in df_acs.columns]
df_acs.columns = [re.sub('[\s+]', '_', col.strip()) for col in df_acs.columns]
# df_acs['percentage'] = round(df_acs['percentage'], 1)

print(df_acs.shape)
df_acs.head(3)

In [ ]:
if sample_type == 'ACS':
    print('Columns: ' + str(list(df_acs.columns)))
    print('Variables: '      + str(unique(df_acs.variable      .values)))
    print('Race/Ethnicity: ' + str(unique(df_acs.race_ethnicity.values)))

if sample_type == 'PUMS':
    print('Columns: ' + str(list(df_acs.columns)))

if indicator_name == 'Cost_6':
    # df_acs = df_acs[(df_acs['housing_type'] == 'Owner') | (df_acs['housing_type'] == 'Renter')]
    df_acs = df_acs[df_acs['housing_type'] == 'Renters/Owners']
    df_acs.loc[df_acs['housing_burden'] == 'Cost burden <=30%'        , 'housing_burden'] = 'Cost burden less than 30 perc'
    df_acs.loc[df_acs['housing_burden'] == 'Cost burden >30% to <=50%', 'housing_burden'] = 'Cost burden 30 to 50 perc'
    df_acs.loc[df_acs['housing_burden'] == 'Cost burden >50%'         , 'housing_burden'] = 'Cost burden greater than 50 perc'


## Data Visualization

In [ ]:
path_plots = os.path.join(path_main, report_theme, sp_folder_out, indicator_name, 'plots')

by_race = True
race_ethnicity = 'rac1p'
by_vars = True
variable = 'housing_burden'
x = 'year'
y = 'percentage'
color = 'rac1p'
line_dash = 'mpo'#None
markers = True
plot_title = 'Housing Burden by Race (%)'
plot_name = 'SACOG vs DRCOG by race'
export = True

In [ ]:

def plot_lines(
    df=df_acs
     , by_vars=by_vars, by_race=by_race, race_ethnicity=race_ethnicity, variable=variable
     , x=x, y=y
     , color=color, line_dash=line_dash, markers=markers
     , plot_title=plot_title, plot_name=plot_name
     , export=export
):
    
    if by_race == True:
        df = df[df[race_ethnicity] != 'All']
    else:
        df = df[df[race_ethnicity] == 'All'].drop(race_ethnicity, axis = 1)

    if by_vars == True:
        vars = unique(df[variable].values)
        for var in vars:
            df2 = df[df[variable] == var]
            fig = px.line(df2, x = x, y = y, color = color, line_dash = line_dash, markers = markers)
            fig.update_layout(title = plot_title + ' - ' + str(var))
            if export == True:
                fig.write_html(os.path.join(path_plots, ''.join([indicator_name + '_', plot_name + '_', var + '_', 'line.html'])))
    else:
        fig = px.line(df, x = x, y = y, color = color, line_dash = line_dash, markers = markers)
        fig.update_layout(title = plot_title)
        if export == True:
            fig.write_html(os.path.join(path_plots, ''.join([indicator_name + '_', plot_name + '_', 'line.html'])))

    return fig.show()
        
plot_lines()

#### Code graveyard

In [ ]:
# path_plots = os.path.join(path_main, report_theme, sp_folder_out, indicator_name, 'plots')
# df = df_acs.copy()

# if by_race == True:
#     df = df[df[race_ethnicity] != 'All']
# else:
#     df = df[df[race_ethnicity] == 'All'].drop('race_ethnicity', axis = 1)
# if by_vars == True:
#     vars = unique(df[variable].values)
#     for var in vars:
#         df2 = df[df[variable] == var]
#         fig = px.line(df2, x = x, y = y, color = color, line_dash = line_dash, markers = markers)
#         fig.update_layout(title = plot_title)
#         # fig.write_html(os.path.join(path_plots, ''.join([indicator_name + '_', plot_name + '_', var + '_', 'line_.html'])))
        
# else:
#     fig = px.line(df, x = x, y = y, color = color, line_dash = line_dash, markers = markers)
#     fig.update_layout(title = plot_title)
#     # fig.write_html(os.path.join(path_plots, ''.join([indicator_name + '_', plot_name + '_', 'line_.html'])))

# fig.show()

In [ ]:
# path_plots = os.path.join(path_main, report_theme, sp_folder_out, indicator_name, 'plots')

# def plot_lines(data, indicator_name, estimate, x, y, group, variables):

#     df_plot = data.copy()
#     color  = group
#     labels = group
#     vars   = unique(variables)
#     x = x
#     y = y
#     vars = unique(df_plot[variables].values)

#     if group == 'race_ethnicity':
#         df_plot = df_plot[df_plot['race_ethnicity'] != 'All']
#         df_plot = df_plot[['year', 'variable', 'race_ethnicity', 'percentage']]
#         df_plot = df_plot[df_plot['race_ethnicity'] != 'All']

#     if group in ['MSA', 'County']:
#         if group == 'County':
#             df_plot = df_plot[['year', 'county_name', 'variable', 'race_ethnicity', 'percentage']]
#         if group == 'MSA':
#             df_plot = df_plot[['year', 'msa', 'variable', 'race_ethnicity', 'percentage']]
#         df_plot = df_plot[df_plot['race_ethnicity'] == 'All'].drop('race_ethnicity', axis = 1)       


#     for var in vars:
#         df_plot2 = df_plot[df_plot['variable'] == var]
        
#         fig = px.line(df_plot2
#                          , x = x
#                          , y = y
#                          , color = color
#                          , markers = True
#                          , labels = group
#                         )
            
#         fig.update_layout(title = var + ' by ' + group)
        
#         fig.write_html(
#             os.path.join(
#                 path_plots
#                 , ''.join([indicator_name + '_'
#                            , var  + '_'
#                            , group + '_'
#                            , estimate  + '_'
#                            , 'line_'
#                            , '.html'])
#             )
#         )
    

In [ ]:
# plot_lines(data = df_acs
#            , indicator_name = indicator_name
#            , estimate = estimate
#            , x = 'year'
#            , y = 'percentage'
#            , group = 'county_name'
#            , variables = 'variable')